# Feature engineering for solar energy forecasting

This notebook loads Calgary municipal solar production data, creates
cyclical month encoding (sin/cos), builds lag and rolling average
features, and visualises seasonal patterns and feature correlations.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, '..')
from src.data_loader import (
    fetch_solar_production, preprocess_production,
    add_rolling_features, add_lag_features,
    compute_facility_stats
)
from src.model import FEATURE_COLUMNS

pd.set_option('display.max_columns', 50)

## 1. Load and preprocess solar data

In [ ]:
raw = fetch_solar_production(force_refresh=False)
df = preprocess_production(raw)

print(f"Records: {len(df)}")
print(f"Facilities: {df['facility_name'].nunique()}")
print(f"Date range: {df['period_dt'].min().date()} to {df['period_dt'].max().date()}")
df.head()

## 2. Cyclical month encoding

Months are cyclical: December (12) is close to January (1), but a
linear encoding would treat them as far apart. The sin/cos
transformation preserves this cyclical structure.

In [ ]:
# The preprocessing function already creates month_sin and month_cos
month_enc = df[['month', 'month_sin', 'month_cos']].drop_duplicates().sort_values('month')
print(month_enc.to_string(index=False))

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=month_enc['month_cos'], y=month_enc['month_sin'],
    mode='markers+text', text=month_enc['month'].astype(str),
    textposition='top center', name='Months'
))

# Draw unit circle
theta = np.linspace(0, 2 * np.pi, 100)
fig.add_trace(go.Scatter(
    x=np.cos(theta), y=np.sin(theta),
    mode='lines', line=dict(dash='dash', color='gray'),
    name='Unit circle'
))

fig.update_layout(
    title='Cyclical month encoding on the unit circle',
    xaxis_title='cos(2pi * month/12)',
    yaxis_title='sin(2pi * month/12)',
    height=400, width=450,
    yaxis=dict(scaleanchor='x')
)
fig.show()

## 3. Seasonal patterns in production

Solar production in Calgary follows a strong seasonal pattern: high
output in summer and low output in winter, driven by solar irradiance
and day length.

In [ ]:
monthly_avg = df.groupby('month')['solar_pv_production_kwh'].mean().reset_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg['month_name'] = monthly_avg['month'].map(lambda m: month_names[m - 1])

fig = px.bar(
    monthly_avg, x='month_name', y='solar_pv_production_kwh',
    title='Average monthly solar production across all facilities',
    labels={'month_name': 'Month', 'solar_pv_production_kwh': 'Avg production (kWh)'}
)
fig.update_layout(height=380)
fig.show()

In [ ]:
# Time series of total monthly production
total_monthly = df.groupby('period_dt')['solar_pv_production_kwh'].sum().reset_index()

fig = px.line(
    total_monthly, x='period_dt', y='solar_pv_production_kwh',
    title='Total monthly solar production over time',
    labels={'period_dt': 'Date', 'solar_pv_production_kwh': 'Total kWh'}
)
fig.update_layout(height=380)
fig.show()

## 4. Lag features

Lag features capture the autocorrelation in production. A facility's
output last month (lag_1m), three months ago (lag_3m), and the same
month last year (lag_12m) are all informative.

In [ ]:
df = add_lag_features(df)

lag_cols = ['lag_1m', 'lag_3m', 'lag_12m']
print("NaN counts in lag features:")
print(df[lag_cols].isna().sum())

# Show one facility's lag values
sample_fac = df['facility_name'].unique()[0]
sample = df[df['facility_name'] == sample_fac].sort_values('period_dt')
sample[['period_dt', 'solar_pv_production_kwh'] + lag_cols].tail(12)

In [ ]:
# Scatter: lag_12m (same month last year) vs current production
clean = df.dropna(subset=['lag_12m'])

fig = px.scatter(
    clean, x='lag_12m', y='solar_pv_production_kwh',
    opacity=0.3,
    title='Same-month-last-year production vs current production',
    labels={'lag_12m': 'Production 12 months ago (kWh)',
            'solar_pv_production_kwh': 'Current production (kWh)'},
    trendline='ols'
)
fig.update_layout(height=400)
fig.show()

## 5. Rolling averages

Rolling averages smooth out month-to-month variability and provide a
measure of recent production trend. We compute 3-month, 6-month, and
12-month rolling averages per facility.

In [ ]:
df = add_rolling_features(df)

roll_cols = ['rolling_avg_3m', 'rolling_avg_6m', 'rolling_avg_12m']
print("Rolling feature sample:")
sample = df[df['facility_name'] == sample_fac].sort_values('period_dt')
sample[['period_dt', 'solar_pv_production_kwh'] + roll_cols].tail(12)

In [ ]:
# Plot rolling averages for one facility
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sample['period_dt'], y=sample['solar_pv_production_kwh'],
    mode='lines', name='Actual', opacity=0.5
))
for col in roll_cols:
    fig.add_trace(go.Scatter(
        x=sample['period_dt'], y=sample[col],
        mode='lines', name=col
    ))

fig.update_layout(
    title=f'Rolling averages for {sample_fac}',
    xaxis_title='Date', yaxis_title='Production (kWh)',
    height=400
)
fig.show()

## 6. Feature correlations

In [ ]:
all_features = FEATURE_COLUMNS + ['solar_pv_production_kwh']
corr = df[all_features].dropna().corr()

fig = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='RdBu_r',
    title='Feature correlation matrix (including target)',
    aspect='auto'
)
fig.update_layout(height=550, width=650)
fig.show()

In [ ]:
# Correlation with target
target_corr = corr['solar_pv_production_kwh'].drop('solar_pv_production_kwh')
target_corr = target_corr.sort_values(ascending=False)
print("Correlation with solar_pv_production_kwh:")
print(target_corr.to_string())

## 7. Facility-level statistics

In [ ]:
stats = compute_facility_stats(df)
print(f"Facility count: {len(stats)}")
stats.sort_values('total_kwh', ascending=False)

## 8. Summary

Feature engineering produced the following predictors for the solar
forecasting model:

| Feature | Description |
|---|---|
| month_sin, month_cos | Cyclical month encoding preserving seasonal continuity |
| year | Captures long-term capacity growth trends |
| lag_1m, lag_3m, lag_12m | Autocorrelation from recent and same-season history |
| rolling_avg_3m, rolling_avg_6m, rolling_avg_12m | Smoothed production trends at different horizons |

The lag_12m feature (same month last year) shows the strongest
correlation with current production, confirming the dominant role of
seasonality. Rolling averages at longer windows track the overall
production trend. The next notebook trains Ridge, Random Forest, and
XGBoost regressors using these features.